# Factory Guardian — Avance 1

Chatbot de consultas por reglas con Python y NLTK. Los cuatro registros y la guía son ficticios. No conecta cámaras, no usa un LLM, no guarda hallazgos y no envía alertas.

El notebook es autocontenido: puede abrirse en Google Colab y ejecutarse de arriba abajo, sin clonar GitHub ni configurar rutas de un equipo. La ejecución conservada en esta entrega fue local; no es evidencia de ejecución en Colab. Cada mensaje es independiente.

In [1]:
%pip install -q "nltk>=3.9,<4"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\rayor\Documents\Codex\2026-09-13\1-lee-cuidadosamente-el-caso-pr\work\factory_guardian_avance1\.venv\Scripts\python.exe -m pip install --upgrade pip


## 1. Base de conocimiento ficticia

Andén A y Pasillo B son zonas válidas sin registros. La guía G-01 no es un procedimiento de planta.

In [2]:
import json
from pathlib import Path

knowledge = json.loads('{\n  "notice": "Base ficticia para una demostración académica; no contiene registros de una planta real.",\n  "zones": ["Cruce C", "Andén A", "Pasillo B"],\n  "events": [\n    {"id": "NM-001", "zone": "Cruce C", "status": "Abierto", "description": "Escenario ficticio de proximidad entre peatón y montacargas."},\n    {"id": "NM-002", "zone": "Cruce C", "status": "En revisión", "description": "Escenario ficticio de visibilidad limitada en el cruce."},\n    {"id": "NM-003", "zone": "Cruce C", "status": "Cerrado", "description": "Escenario ficticio de invasión de la ruta peatonal."},\n    {"id": "NM-004", "zone": "Cruce C", "status": "Abierto", "description": "Escenario ficticio de interacción en una zona de paso compartida."}\n  ],\n  "guides": [\n    {"id": "G-01", "zone": "Cruce C", "text": "Guía demostrativa G-01, ficticia: consultar la señalización y la segregación del cruce en el procedimiento aprobado de la planta y solicitar revisión al responsable de seguridad. Esta guía no sustituye un procedimiento real ni una evaluación profesional."}\n  ]\n}\n')
Path('knowledge.json').write_text(json.dumps(knowledge, ensure_ascii=False, indent=2), encoding='utf-8')
print(knowledge['notice'])
print('Registros ficticios:', len(knowledge['events']))

Base ficticia para una demostración académica; no contiene registros de una planta real.
Registros ficticios: 4


## 2. Chatbot

Este código es el mismo módulo `bot.py`. NLTK tokeniza; reglas explícitas consultan el JSON. No se entrena un modelo.

In [3]:
"""Consultas deterministas con NLTK y una base de conocimiento ficticia.

Cada mensaje es independiente. No se registran eventos ni se envían alertas.
"""
from dataclasses import dataclass
import json
from pathlib import Path
import re
import unicodedata

from nltk.tokenize import wordpunct_tokenize


@dataclass(frozen=True)
class Reply:
    """Intención identificada y respuesta que puede mostrar cualquier interfaz."""
    intent: str
    text: str


def normalize(text: str) -> str:
    """Ignora mayúsculas y acentos sin descargar modelos de lenguaje."""
    return "".join(
        char for char in unicodedata.normalize("NFD", text.casefold())
        if unicodedata.category(char) != "Mn"
    )


def tokenize(text: str) -> tuple[str, ...]:
    """La tokenización NLTK produce las unidades usadas por las reglas."""
    return tuple(token for token in wordpunct_tokenize(normalize(text)) if token.isalnum())


def load_knowledge(path=None) -> dict:
    """Carga JSON local; la ausencia de una fuente no se disimula con datos inventados."""
    source = Path(path) if path is not None else Path(__file__).with_name("knowledge.json")
    return json.loads(source.read_text(encoding="utf-8"))


class FactoryGuardianBot:
    """Chatbot de solo consultas con ocho intenciones y reglas ordenadas."""

    def __init__(self, knowledge_path=None):
        self.knowledge = load_knowledge(knowledge_path)

    def _zone(self, tokens):
        words = " " + " ".join(tokens) + " "
        return next((zone for zone in self.knowledge["zones"]
                     if " " + " ".join(tokenize(zone)) + " " in words), None)

    def _ask_zone(self):
        return ("Escribe la consulta completa con una zona válida: "
                + ", ".join(self.knowledge["zones"])
                + ". La zona está ausente o no se reconoce. No conservo contexto entre mensajes.")

    def respond(self, text: str) -> Reply:
        """Resuelve primero las consultas útiles; después ayuda, cierre y saludo."""
        tokens = tokenize(text)
        words = set(tokens)
        zone = self._zone(tokens)
        identifier = re.search(r"\bnm\s*-?\s*(\d{3})\b", normalize(text))

        if words & {"alcance", "camara", "camaras", "llm", "alerta", "alertas",
                    "registrar", "guardar", "detectar"}:
            return Reply("alcance", "Este avance ofrece solo consultas sobre una base de ejemplo. "
                         "No conecto cámaras, no envío alertas, no registro eventos y no uso un LLM. "
                         "No sustituyo la revisión humana.")

        if identifier or words & {"estado", "estatus"}:
            if identifier is None:
                return Reply("consultar_estado", "Indica el ID completo en la consulta, por ejemplo: "
                             "¿Cuál es el estado de NM-001? No conservo contexto entre mensajes.")
            event_id = "NM-" + identifier.group(1)
            event = next((item for item in self.knowledge["events"] if item["id"] == event_id), None)
            if event is None:
                return Reply("consultar_estado", f"El ID {event_id} no existe en la base de conocimiento.")
            return Reply("consultar_estado", f"{event_id}: {event['status']}. Zona: {event['zone']}.")

        if words & {"procedimiento", "procedimientos", "guia", "guias"}:
            if zone is None:
                return Reply("consultar_procedimiento", self._ask_zone())
            guide = next((item for item in self.knowledge["guides"] if item["zone"] == zone), None)
            if guide is None:
                return Reply("consultar_procedimiento", f"No hay guía demostrativa para {zone}. "
                             "Consulta el procedimiento aprobado con el responsable de seguridad.")
            return Reply("consultar_procedimiento", guide["text"])

        if words & {"evento", "eventos", "incidente", "incidentes", "near", "miss", "cuantos", "cuantas"}:
            if zone is None:
                return Reply("consultar_eventos", self._ask_zone())
            count = sum(item["zone"] == zone for item in self.knowledge["events"])
            noun = "registro disponible" if count == 1 else "registros disponibles"
            return Reply("consultar_eventos", f"{zone} tiene {count} {noun}.")

        if words & {"ayuda", "opciones", "consultas"}:
            return Reply("ayuda", "Puedo consultar eventos por zona, el estado de un ID y una guía demostrativa. "
                         "Ejemplos: 'Eventos en Cruce C', 'Estado NM-001', 'Guía de Cruce C'. "
                         "Envía cada consulta completa.")
        if words & {"adios", "hasta", "salir", "chau"}:
            return Reply("despedida", "Hasta luego. Las consultas de esta demostración no sustituyen la revisión humana.")
        if words & {"hola", "buenos", "buenas", "saludos"}:
            return Reply("saludo", "Hola, soy Factory Guardian. Consulto una base de ejemplo por reglas. "
                         "Escribe 'ayuda' para conocer las consultas disponibles.")
        return Reply("no_reconocida", "No reconozco esa consulta. Escribe 'ayuda' y reformula la pregunta completa "
                     "con la zona o el ID correspondiente.")


## 3. Evaluador

Compara respuestas reales con expectativas escritas previamente; conserva JSON y Markdown.

In [4]:
"""Evidencia reproducible: expectativas declaradas frente a respuestas reales."""
from datetime import datetime, timezone
import importlib.metadata
import json
from pathlib import Path
import platform
import sys


def run_sessions(bot, cases):
    """Ejecuta cada entrada y contrasta intención y fragmentos escritos a mano."""
    sessions = []
    for case in cases:
        turns = []
        for case_turn in case["turns"]:
            reply = bot.respond(case_turn["input"])
            missing = [part for part in case_turn["expected_contains"]
                       if part.casefold() not in reply.text.casefold()]
            passed = reply.intent == case_turn["expected_intent"] and not missing
            turns.append({**case_turn, "actual_intent": reply.intent,
                          "actual_text": reply.text, "missing_fragments": missing, "passed": passed})
        sessions.append({"id": case["id"], "purpose": case["purpose"], "turns": turns})
    all_turns = [turn for session in sessions for turn in session["turns"]]
    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "execution_environment": "google_colab" if "google.colab" in sys.modules else "local",
        "data_scope": "Escenarios y registros ficticios; no son resultados industriales.",
        "context_policy": "Sin estado entre mensajes; cada consulta debe ser completa.",
        "runtime": {"python": platform.python_version(), "platform": platform.system(),
                    "nltk": importlib.metadata.version("nltk")},
        "summary": {"sessions": len(sessions), "turns": len(all_turns),
                    "passed_turns": sum(turn["passed"] for turn in all_turns),
                    "failed_turns": sum(not turn["passed"] for turn in all_turns)},
        "sessions": sessions,
    }


def save_results(result, output_dir):
    """Conserva transcripciones reales en JSON y Markdown legible."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    json_path = output / "conversations.json"
    markdown_path = output / "conversations.md"
    json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    summary = result["summary"]
    lines = ["# Diez conversaciones de prueba de Factory Guardian", "",
             "Escenarios ficticios de evaluación; las respuestas siguientes fueron ejecutadas.",
             f"Entorno: {result['execution_environment']}. Python: {result['runtime']['python']}. "
             f"NLTK: {result['runtime']['nltk']}.",
             f"Sesiones: {summary['sessions']}; mensajes: {summary['turns']}; "
             f"aprobados: {summary['passed_turns']}; fallidos: {summary['failed_turns']}.",
             "Sin memoria entre turnos: los seguimientos contienen la consulta completa.", ""]
    for session in result["sessions"]:
        lines += [f"## {session['id']}: {session['purpose']}", ""]
        for index, turn in enumerate(session["turns"], 1):
            lines += [f"{index}. Usuario: {turn['input']}", "",
                      f"   Bot [{turn['actual_intent']}]: {turn['actual_text']}", "",
                      f"   Esperado: {turn['expected_intent']}; contiene {turn['expected_contains']}. "
                      f"Resultado: {'APROBADO' if turn['passed'] else 'FALLIDO'}.", ""]
    markdown_path.write_text("\n".join(lines), encoding="utf-8")
    return json_path, markdown_path


## 4. Diez conversaciones distintas

Cada sesión tiene dos mensajes de usuario. Los seguimientos vuelven a indicar la zona o el ID porque no hay memoria entre turnos. Aprobar estos casos no demuestra comprensión general del lenguaje ni utilidad industrial.

In [5]:
cases = json.loads('[\n  {"id":"C01","purpose":"Presentación y orientación","turns":[\n    {"input":"Hola","expected_intent":"saludo","expected_contains":["Factory Guardian","base de ejemplo"]},\n    {"input":"Necesito ayuda","expected_intent":"ayuda","expected_contains":["eventos","estado","guía"]}\n  ]},\n  {"id":"C02","purpose":"Contrastar una zona con registros y una zona vacía","turns":[\n    {"input":"¿Cuántos near-miss hay en el Cruce C?","expected_intent":"consultar_eventos","expected_contains":["4 registros","Cruce C"]},\n    {"input":"¿Cuántos eventos hay en Andén A?","expected_intent":"consultar_eventos","expected_contains":["0 registros","Andén A"]}\n  ]},\n  {"id":"C03","purpose":"Prioridad de consulta útil y normalización","turns":[\n    {"input":"Hola, ¿CUÁNTOS EVENTOS hay en CRUCE C?","expected_intent":"consultar_eventos","expected_contains":["4 registros","Cruce C"]},\n    {"input":"Muéstrame eventos en PASILLO B.","expected_intent":"consultar_eventos","expected_contains":["0 registros","Pasillo B"]}\n  ]},\n  {"id":"C04","purpose":"Estados de dos identificadores existentes","turns":[\n    {"input":"¿Cuál es el estado de NM-001?","expected_intent":"consultar_estado","expected_contains":["NM-001","Abierto"]},\n    {"input":"Consulta estado de NM-003","expected_intent":"consultar_estado","expected_contains":["NM-003","Cerrado"]}\n  ]},\n  {"id":"C05","purpose":"Identificador inexistente y reformulación completa","turns":[\n    {"input":"Estado de NM-999","expected_intent":"consultar_estado","expected_contains":["NM-999","no existe"]},\n    {"input":"Estado NM-002","expected_intent":"consultar_estado","expected_contains":["NM-002","En revisión"]}\n  ]},\n  {"id":"C06","purpose":"Consulta de eventos sin zona","turns":[\n    {"input":"¿Cuántos eventos hay?","expected_intent":"consultar_eventos","expected_contains":["consulta completa","zona","Cruce C"]},\n    {"input":"¿Cuántos eventos hay en Cruce C?","expected_intent":"consultar_eventos","expected_contains":["4 registros","Cruce C"]}\n  ]},\n  {"id":"C07","purpose":"Consulta de estado sin identificador","turns":[\n    {"input":"Necesito saber el estado","expected_intent":"consultar_estado","expected_contains":["ID completo","NM-001"]},\n    {"input":"¿Cuál es el estado de NM-004?","expected_intent":"consultar_estado","expected_contains":["NM-004","Abierto"]}\n  ]},\n  {"id":"C08","purpose":"Guía disponible y zona sin guía","turns":[\n    {"input":"Consulta el procedimiento del Cruce C","expected_intent":"consultar_procedimiento","expected_contains":["G-01","demostrativa","no sustituye"]},\n    {"input":"¿Tienes una guía para Andén A?","expected_intent":"consultar_procedimiento","expected_contains":["No hay guía","Andén A"]}\n  ]},\n  {"id":"C09","purpose":"Límites de cámaras, alertas y registro","turns":[\n    {"input":"¿Puedes conectar una cámara y enviar alertas?","expected_intent":"alcance","expected_contains":["solo consultas","No conecto cámaras","no envío alertas"]},\n    {"input":"Quiero registrar un nuevo evento","expected_intent":"alcance","expected_contains":["no registro","base de ejemplo"]}\n  ]},\n  {"id":"C10","purpose":"Solicitud fuera del dominio y cierre","turns":[\n    {"input":"¿Qué clima hará mañana?","expected_intent":"no_reconocida","expected_contains":["No reconozco","ayuda"]},\n    {"input":"Adiós","expected_intent":"despedida","expected_contains":["Hasta luego"]}\n  ]}\n]\n')
bot = FactoryGuardianBot('knowledge.json')
results = run_sessions(bot, cases)
save_results(results, 'artifacts')
for session in results['sessions']:
    print('\n' + session['id'] + ': ' + session['purpose'])
    for turn in session['turns']:
        print('Usuario:', turn['input'])
        print('Bot [' + turn['actual_intent'] + ']:', turn['actual_text'])
        print('Resultado:', 'APROBADO' if turn['passed'] else 'FALLIDO')
print('\nResumen:', results['summary'])
print('Entorno real:', results['execution_environment'])
assert results['summary']['sessions'] == 10
assert results['summary']['failed_turns'] == 0


C01: Presentación y orientación
Usuario: Hola
Bot [saludo]: Hola, soy Factory Guardian. Consulto una base de ejemplo por reglas. Escribe 'ayuda' para conocer las consultas disponibles.
Resultado: APROBADO
Usuario: Necesito ayuda
Bot [ayuda]: Puedo consultar eventos por zona, el estado de un ID y una guía demostrativa. Ejemplos: 'Eventos en Cruce C', 'Estado NM-001', 'Guía de Cruce C'. Envía cada consulta completa.
Resultado: APROBADO

C02: Contrastar una zona con registros y una zona vacía
Usuario: ¿Cuántos near-miss hay en el Cruce C?
Bot [consultar_eventos]: Cruce C tiene 4 registros disponibles.
Resultado: APROBADO
Usuario: ¿Cuántos eventos hay en Andén A?
Bot [consultar_eventos]: Andén A tiene 0 registros disponibles.
Resultado: APROBADO

C03: Prioridad de consulta útil y normalización
Usuario: Hola, ¿CUÁNTOS EVENTOS hay en CRUCE C?
Bot [consultar_eventos]: Cruce C tiene 4 registros disponibles.
Resultado: APROBADO
Usuario: Muéstrame eventos en PASILLO B.
Bot [consultar_eventos]: 

## 5. Probar otra consulta

Modifica la variable `consulta` y ejecuta la celda. Para consultas incompletas el chatbot solicita reformular; no inventa el parámetro faltante.

In [6]:
consulta = 'Eventos en Cruce C'
respuesta = bot.respond(consulta)
print(respuesta.intent, '→', respuesta.text)

consultar_eventos → Cruce C tiene 4 registros disponibles.


## Límites

Las reglas reconocen vocabulario acotado; no resuelven referencias como ‘el primero’, negaciones complejas ni varias peticiones simultáneas. Las cifras proceden de cuatro registros de demostración. La visión futura de cámara y LLM requiere otro desarrollo y validación humana.